# RouteKnower on Colab: can a learned correction beat Passio's arrival times?

[Open in Colab](https://colab.research.google.com/github/blobberus/straight-bussing/blob/main/experiments/routeknower/colab/routeknower_colab.ipynb)
· Plain-English explanation: [PLAIN-ENGLISH.ipynb](https://colab.research.google.com/github/blobberus/straight-bussing/blob/main/experiments/routeknower/colab/PLAIN-ENGLISH.ipynb)
· Technical notes: `TECHNICAL.md` in the same folder.

**In plain words.** Straight Bussing records, every minute, what Passio's feed told riders about each upcoming
bus and when that bus really came. This notebook downloads those public files, checks that it scores them exactly
like the project does, then tries the cheapest ways to make Passio's times better: a learned per-stop shift, and
a small tree model (LightGBM). A deep network is included as an **optional add-on, off by default**: with the few
weeks of data so far it is not expected to beat the trees, and anything that ships must run on the phone in
plain JavaScript or Swift. Press **Runtime > Run all**. The free Colab tier is enough; it takes a few minutes.

**The goal it is judged on** (the owner's rule): a prediction made L minutes ahead should be right within 10% of
L, and almost never too late (a bus that comes earlier than shown is a missed bus). The working target is
**band B**: from 10% of L too early to 1% of L too late, without being too late more often than Passio.

## 1. Settings
`FAST` is a quick look (a quarter of the arrivals, fewer trees). `FULL` uses everything and is still a few
minutes on the free tier. Leave `RUN_DEEP` off unless you want to see the deep model for yourself.

In [ ]:
MODE = "FULL"          # "FAST" or "FULL"
DAYS = None            # None = every archived day with Passio predictions; or e.g. ["2026-10-13", "2026-10-14"]
RUN_DEEP = False       # the optional deep model (PyTorch); uses the GPU if you picked one
USE_DRIVE = False      # True: keep downloads and results in your Google Drive (Colab asks for permission)
SEED = 7

## 2. Setup
Installs anything missing (Colab already has pandas, LightGBM and PyTorch) and starts a timer, so every step
prints how long it took.

In [ ]:
import gzip, json, math, os, random, subprocess, sys, time, urllib.request
from pathlib import Path


def need(module, pip_name=None):
    try:
        return __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])
        return __import__(module)


np = need("numpy")
pd = need("pandas")
lgb = need("lightgbm")
try:
    import psutil
except ImportError:
    psutil = None

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content") if Path("/content").exists() else Path.cwd()
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/routeknower")
CACHE, OUT = BASE / "routeknower_data", BASE / "routeknower_out"
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
FAST = MODE.upper() == "FAST"
random.seed(SEED)
np.random.seed(SEED)

T0, _last, PEAK, TIMES = time.time(), [time.time()], [0.0], {}


def step(name):
    """Print the time of the step that just ended and the memory in use."""
    now = time.time()
    rss = psutil.Process().memory_info().rss / 2**20 if psutil else float("nan")
    PEAK[0] = max(PEAK[0], rss)
    TIMES[name] = round(now - _last[0], 1)
    print(f"[{now - T0:6.1f} s] {name}: {now - _last[0]:.1f} s, memory {rss:,.0f} MB")
    _last[0] = now


print("Colab" if IN_COLAB else "local", "| mode", MODE, "| files in", BASE)
step("setup")

## 3. Download the public data
Only public files, no login: the daily forecast tables `eta-<day>.csv.gz` (release assets of the project's GitHub
repo) and the daily score summary `data/eta/summary.csv` (the repo's `data` branch). Files are cached; a table
that was rebuilt on GitHub (different size) is downloaded again.

In [ ]:
REPO = "blobberus/straight-bussing"
UA = {"User-Agent": "routeknower-colab"}


def get(url, timeout=120):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout).read()


(CACHE / "summary.csv").write_bytes(get(f"https://raw.githubusercontent.com/{REPO}/data/data/eta/summary.csv"))
summary = pd.read_csv(CACHE / "summary.csv", dtype={"service_day": str, "route": str})
passio_days = sorted(summary.loc[(summary.source == "passio") & (summary.n > 0), "service_day"].unique())
releases = json.loads(get(f"https://api.github.com/repos/{REPO}/releases?per_page=100"))
assets = {a["name"]: a for r in releases if r["tag_name"].startswith("data-archive-") for a in r["assets"]}
days = []
for d in sorted(DAYS) if DAYS else passio_days:
    a = assets.get(f"eta-{d}.csv.gz")
    if a is None:
        print("not archived yet:", d)
        continue
    path = CACHE / a["name"]
    if not path.exists() or path.stat().st_size != a["size"]:
        path.write_bytes(get(a["browser_download_url"], timeout=600))
    days.append(d)
print(len(days), "days with Passio predictions:", ", ".join(days))
assert len(days) >= 2, "Need at least 2 days to learn on one and test on the next."
step("download")

## 4. Load the tables
One row = one minute, one bus, one of its arrivals in the next 45 minutes: what Passio showed then, what the
timetable said, where the bus was, and when it really came. Compact types keep memory low.

In [ ]:
STR = ["service_day", "day_type", "vehicle_id", "route_id", "stop_id", "label_status", "passio_status"]
NUMS = ["query_epoch", "dow", "hour", "stop_index", "visit_rank", "next_trip", "stops_away", "dist_m",
        "crosses_terminal", "last_age_s", "speed_mps", "standing_s", "delay_s", "actual_epoch", "lead_s",
        "passio_epoch", "passio_age_s", "passio_err_s", "tt_epoch", "tt_err_s", "ttd_epoch", "ttd_err_s"]


def load(d):
    t = pd.read_csv(CACHE / f"eta-{d}.csv.gz", usecols=STR + NUMS, dtype={c: "string" for c in STR}, low_memory=False)
    t["service_day"] = d
    return t


df = pd.concat([load(d) for d in days], ignore_index=True)       # file order kept: the scoring below relies on it
for c in NUMS:
    if c not in ("query_epoch", "actual_epoch", "passio_epoch", "tt_epoch", "ttd_epoch"):
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
for c in ("route_id", "stop_id", "day_type", "label_status", "passio_status", "service_day"):
    df[c] = df[c].astype("category")
print(f"{len(df):,} rows, {df.memory_usage(deep=True).sum() / 2**20:,.0f} MB")
step("load")

## 5. Score exactly like the project, and check it
The project scores **one sample per arrival and lead bin** (the row whose lead is nearest the middle of the bin,
never an average over minutes), sets **gross misses** aside (more than 10 minutes or half the lead off; still
counted), and reports the share within 10% of the lead and the share **too late** by more than 1% of the lead.
Here we recompute Passio's and the timetable's numbers from the tables and compare them with the project's own
summary. If they differ, the notebook stops: everything after this point depends on scoring the same way.

In [ ]:
BINS = [(0, 360, "0-6"), (360, 600, "6-10"), (600, 900, "10-15"), (900, 1200, "15-20"), (1200, 1800, "20-30"),
        (1800, 2701, "30-45")]
BIN_NAMES = [b[2] for b in BINS]
MID = {b: (lo + min(hi, 2700)) / 2 for lo, hi, b in BINS}
df["bin"] = pd.cut(df["lead_s"], [b[0] for b in BINS] + [2701], right=False, labels=BIN_NAMES)
ARR = ["vehicle_id", "stop_id", "actual_epoch", "bin"]          # one arrival in one lead bin


def picks(frame, has):
    """The project's sample: per arrival and bin, the first row nearest the bin middle among rows where `has`."""
    f = frame[(frame.label_status == "arrived") & (frame.lead_s > 0) & frame["bin"].notna() & has]
    dist = (f["lead_s"] - f["bin"].map(MID).astype("float32")).abs()
    return f.loc[dist.groupby([f[c] for c in ARR], observed=True, sort=False).idxmin().values]


def score(e, lead):
    """Counts the project uses, plus band B (10% early to 1% late) and MAE, for errors e = predicted - actual."""
    e, lead = np.asarray(e, dtype="float64"), np.asarray(lead, dtype="float64")
    gross = np.abs(e) > np.maximum(600.0, 0.5 * lead)
    ok, ae = ~gross, np.abs(e)
    return {"n": int(len(e)), "gross": int(gross.sum()), "pass_10": int((ae[ok] <= 0.10 * lead[ok]).sum()),
            "pass_10_floor": int((ae[ok] <= np.maximum(60.0, 0.10 * lead[ok])).sum()),
            "band_b": int(((e[ok] >= -0.10 * lead[ok]) & (e[ok] <= 0.01 * lead[ok])).sum()),
            "late_1": int((e > 0.01 * lead).sum()), "early_10": int((e < -0.10 * lead).sum()),
            "abs_err_sum": float(ae[ok].sum())}


SOURCES = {"passio": "passio_err_s", "timetable": "tt_err_s", "timetable_delay": "ttd_err_s"}
mismatch = []
for src, col in SOURCES.items():
    p = picks(df, df[col].notna())
    for (d, b), g in p.groupby(["service_day", "bin"], observed=True):
        mine = score(g[col], g["lead_s"])
        ref = summary[(summary.service_day == d) & (summary.source == src) & (summary.lead_bin == b)
                      & (summary.route == "ALL") & (summary.segment == "all")]
        if ref.empty:
            mismatch.append((d, src, b, "missing in summary.csv"))
            continue
        for k in ("n", "gross", "pass_10", "pass_10_floor", "late_1", "early_10"):
            if int(ref[k].iloc[0]) != mine[k]:
                mismatch.append((d, src, b, k, int(ref[k].iloc[0]), mine[k]))
if mismatch:
    print("First differences:", mismatch[:10])
    raise SystemExit("STOP: this notebook does not score like data/eta/summary.csv. Fix the scoring before "
                     "trusting any result (or the tables and the summary are from different rebuilds).")
print(f"Scoring check passed: {len(days)} days x 3 sources x {len(BINS)} bins match data/eta/summary.csv exactly.")
step("scoring check")

## 6. What the models may use
Only what is known **at the minute of the question**: route, stop, hour, weekday, where the bus is (stops and
metres to go, standing time, how old its last stop is), its delay against the timetable, and what Passio and the
timetable say right now (as "minutes from now"). Never the real arrival, the real lead, or any error column.

The models learn a **shift**: how many seconds to add to Passio's time. A tree model or network predicts a low
**quantile** of that shift (for example "80% of the time the bus comes at least this much later than Passio
shows"), because coming out too late is far worse than coming out a little early. Which quantile is used is
chosen per lead bin on the last learning day, keeping "too late" no higher than Passio's.

In [ ]:
df["passio_lead"] = (df["passio_epoch"] - df["query_epoch"]).astype("float32")     # what Passio shows, in seconds
df["tt_lead"] = (df["tt_epoch"] - df["query_epoch"]).astype("float32")
df["ttd_lead"] = (df["ttd_epoch"] - df["query_epoch"]).astype("float32")
df["weekend"] = (df["day_type"] != "weekday").astype("float32")
df["y"] = (df["actual_epoch"] - df["passio_epoch"]).astype("float32")             # the shift to learn (seconds)
df["pbin"] = pd.cut(df["passio_lead"].clip(lower=0), [b[0] for b in BINS] + [1e9], right=False,
                    labels=BIN_NAMES)                                              # lead bin as Passio shows it
CAT = ["route_id", "stop_id"]
NUM = ["dow", "hour", "weekend", "stop_index", "visit_rank", "next_trip", "stops_away", "dist_m",
       "crosses_terminal", "last_age_s", "speed_mps", "standing_s", "delay_s", "passio_age_s", "passio_lead",
       "tt_lead", "ttd_lead"]
FEATURES = CAT + NUM
LEAK = {"actual_epoch", "lead_s", "label_status", "passio_err_s", "tt_err_s", "ttd_err_s", "y", "bin"}
assert not LEAK & set(FEATURES), "a label column slipped into the features"
train_mask = (df.label_status == "arrived") & (df.passio_status == "shown") & (df.lead_s > 0) & (df.lead_s <= 2700)
pool = df[train_mask & df["pbin"].notna()].copy()
if FAST:   # a quarter of the arrivals (whole arrivals, so no arrival is split)
    keep = pd.util.hash_pandas_object(pool[["vehicle_id", "stop_id", "actual_epoch"]], index=False) % 4 == 0
    pool = pool[keep.values]
print(f"{len(pool):,} rows where Passio showed a time; features: {', '.join(FEATURES)}")
step("features")

## 7. The models
1. **Passio as shown** (the bar to beat).
2. **Per-stop shift**: for each route, stop, lead bin and "this trip or the next", a quantile of the shift on the
   learning days (backs off to route, then lead bin, when a stop has few samples). This is a lookup table: it
   could ship in the app as a small JSON file today.
3. **Trees** (LightGBM, quantile objective, CPU): the same idea, but it can combine every feature.
4. **Deep** (optional, `RUN_DEEP`): a small network with learned embeddings for route, stop, hour, weekday and
   lead bin, predicting several quantiles at once.

In [ ]:
ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5]
ROUNDS = 150 if FAST else 400


SAFETY = 0.9   # a shift is only used where it was too late at most 90% as often as Passio on the choosing day


def choose(pred_by_alpha, frame):
    """Per Passio lead bin: the quantile with the best band B on `frame` that was too late at most SAFETY times as
    often as Passio there; "none" (keep Passio's time) when no quantile is that safe or it would not gain.
    frame has y, lead_s, pbin. The margin is there because one day's too-late share does not carry over exactly."""
    best = {}
    y = frame["y"].to_numpy()
    for b in BIN_NAMES:
        m = (frame["pbin"] == b).to_numpy()
        if m.sum() < 30:
            best[b] = "none"
            continue
        lead = frame["lead_s"].to_numpy()[m]
        p = score(-y[m], lead)                     # Passio's own error = passio - actual = -y
        p_late, p_band = p["late_1"] / p["n"], p["band_b"] / max(1, p["n"] - p["gross"])
        pick, pick_band = "none", p_band
        for a, c in pred_by_alpha.items():
            s = score(c[m] - y[m], lead)
            band = s["band_b"] / max(1, s["n"] - s["gross"])
            if s["late_1"] / s["n"] <= SAFETY * p_late and band > pick_band:
                pick, pick_band = a, band
        best[b] = pick
    return best


def apply_choice(pred_by_alpha, frame, choice):
    out = np.zeros(len(frame), dtype="float64")            # "none" = shift 0 = Passio's own time
    pb = frame["pbin"].astype(str).to_numpy()
    for b, a in choice.items():
        if a != "none":
            m = pb == b
            out[m] = pred_by_alpha[a][m]
    return out


def used(choice):
    return {a for a in choice.values() if a != "none"}


# ---- 2. per-stop shift (a lookup table)
LEVELS = [["route_id", "stop_id", "next_trip", "pbin"], ["route_id", "next_trip", "pbin"], ["pbin"]]


def shift_fit(tr, alpha):
    return [tr.groupby(k, observed=True)["y"].agg([("q", lambda s: s.quantile(alpha)), ("n", "size")]) for k in LEVELS]


def shift_predict(tables, frame, min_n=30):
    out = np.full(len(frame), np.nan)
    for keys, t in zip(LEVELS, tables):
        t = t[t["n"] >= min_n]
        j = frame[keys].merge(t["q"].rename("q").reset_index(), on=keys, how="left")["q"].to_numpy()
        out = np.where(np.isnan(out), j, out)
    return np.nan_to_num(out, nan=0.0)


# ---- 3. trees
def tree_fit(tr, alpha, va=None, rounds=ROUNDS):
    params = dict(objective="quantile", alpha=alpha, learning_rate=0.05, num_leaves=31, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1,
                  seed=SEED, num_threads=os.cpu_count() or 2)
    dtr = lgb.Dataset(tr[FEATURES], tr["y"], categorical_feature=CAT, free_raw_data=False)
    if va is None:
        return lgb.train(params, dtr, num_boost_round=rounds)
    dva = lgb.Dataset(va[FEATURES], va["y"], categorical_feature=CAT, reference=dtr)
    return lgb.train(params, dtr, num_boost_round=rounds, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(30, verbose=False)])


step("model code")

### Optional: the deep model (off by default)
Turn on `RUN_DEEP` in the settings to try it. It is built small on purpose (two layers, embeddings of 4 to 16
numbers), uses the GPU with mixed precision when there is one, and stops early when the last learning day stops
improving. Why it is off: with a few weeks of data a network has too few days to learn anything the trees
cannot, it costs more to run, and a network is harder to ship inside the app than a lookup table or a small tree
file. Revisit it once there are about 8 weeks of data (a full quarter with a break week) or a feature only a
network can use well, such as the sequence of the bus's last stops.

In [ ]:
if RUN_DEEP:
    torch = need("torch")
    import torch.nn as nn

    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    QS = ALPHAS
    EMB = {"route_id": 8, "stop_id": 16, "hour": 4, "dow": 4, "pbin": 4}
    DNUM = [c for c in NUM if c not in ("hour", "dow")]
    vocab = {c: {v: i + 1 for i, v in enumerate(pool[c].astype(str).unique())} for c in EMB}

    def tensors(frame, stats=None):
        cats = np.stack([frame[c].astype(str).map(vocab[c]).fillna(0).to_numpy(dtype="int64") for c in EMB], 1)
        x = frame[DNUM].to_numpy(dtype="float32")
        miss = np.isnan(x).astype("float32")
        if stats is None:
            stats = (np.nanmean(x, 0), np.nanstd(x, 0) + 1e-6)
        x = np.nan_to_num((x - stats[0]) / stats[1])
        return torch.tensor(cats), torch.tensor(np.concatenate([x, miss], 1)), stats

    class Net(nn.Module):
        def __init__(self, n_num):
            super().__init__()
            self.emb = nn.ModuleList([nn.Embedding(len(vocab[c]) + 1, d) for c, d in EMB.items()])
            width = sum(EMB.values()) + n_num
            self.body = nn.Sequential(nn.Linear(width, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 64), nn.ReLU())
            self.head = nn.Linear(64, len(QS))

        def forward(self, cats, num):
            h = torch.cat([e(cats[:, i]) for i, e in enumerate(self.emb)] + [num], 1)
            out = self.head(self.body(h))
            # quantiles in order: the first, then non-negative steps
            return torch.cumsum(torch.cat([out[:, :1], nn.functional.softplus(out[:, 1:])], 1), 1)

    def pinball(pred, y):
        q = torch.tensor(QS, device=pred.device).view(1, -1)
        d = y.view(-1, 1) - pred
        return torch.maximum(q * d, (q - 1) * d).mean()

    def deep_fit(tr, va, epochs=4 if FAST else 15, batch=4096):
        torch.manual_seed(SEED)
        ct, xt, stats = tensors(tr)
        cv, xv, _ = tensors(va, stats)
        yt = torch.tensor(tr["y"].to_numpy(dtype="float32") / 600.0)         # in units of 10 minutes
        yv = torch.tensor(va["y"].to_numpy(dtype="float32") / 600.0)
        net = Net(xt.shape[1]).to(DEV)
        opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
        best, best_state, bad = 1e9, None, 0
        for ep in range(epochs):
            net.train()
            perm = torch.randperm(len(yt))
            for i in range(0, len(yt), batch):
                j = perm[i:i + batch]
                with torch.autocast(device_type=DEV, enabled=DEV == "cuda"):
                    loss = pinball(net(ct[j].to(DEV), xt[j].to(DEV)).float(), yt[j].to(DEV))
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            net.eval()
            with torch.no_grad():
                vl = float(pinball(net(cv.to(DEV), xv.to(DEV)).float(), yv.to(DEV)))
            if vl < best - 1e-4:
                best, best_state, bad = vl, {k: v.detach().clone() for k, v in net.state_dict().items()}, 0
            else:
                bad += 1
                if bad >= 3:
                    break
        net.load_state_dict(best_state)
        return net, stats

    def deep_predict(model, frame):
        net, stats = model
        c, x, _ = tensors(frame, stats)
        net.eval()
        with torch.no_grad():
            out = torch.cat([net(c[i:i + 65536].to(DEV), x[i:i + 65536].to(DEV)).float().cpu()
                             for i in range(0, len(c), 65536)]).numpy() * 600.0
        return {a: out[:, k] for k, a in enumerate(QS)}

    print("deep model on", DEV)
    step("deep model code")

## 8. Learn on earlier days, test on the next one
Rolling origin by **whole service days**: learn on days 1 to k, test on day k + 1, for every k from 2 on. Never a
random split, and one arrival is never in both. Each model's quantile per lead bin is chosen on the last learning
day (so the first test day is day 3: two learning days are the minimum to choose safely). Where no quantile is
safe enough, the model keeps Passio's time. Every model is scored on **the same rows**: the project's sample for
Passio on the test day.

In [ ]:
def passio_sample(day):
    rows = picks(df[df.service_day == day], (df.passio_status == "shown")[df.service_day == day])
    return rows[rows["pbin"].notna()]


def fit_all(tr_days, test_day):
    """Fit every model on tr_days (choosing quantiles on the last of them) and predict test_day's Passio sample."""
    tr = pool[pool.service_day.isin(tr_days)]
    fit_tr, va = tr[tr.service_day.isin(tr_days[:-1])], tr[tr.service_day == tr_days[-1]]
    va_rows, te_rows = passio_sample(tr_days[-1]), passio_sample(test_day)
    preds, choices = {"passio": np.zeros(len(te_rows))}, {}
    # per-stop shift
    tabs = {a: shift_fit(fit_tr, a) for a in ALPHAS}
    ch = choose({a: shift_predict(tabs[a], va_rows) for a in ALPHAS}, va_rows)
    full = {a: shift_fit(tr, a) for a in used(ch)}
    preds["stop_shift"] = apply_choice({a: shift_predict(full[a], te_rows) for a in full}, te_rows, ch)
    choices["stop_shift"] = ch
    # trees
    models = {a: tree_fit(fit_tr, a, va) for a in ALPHAS}
    ch = choose({a: models[a].predict(va_rows[FEATURES]) for a in ALPHAS}, va_rows)
    final = {a: tree_fit(tr, a, rounds=max(50, models[a].best_iteration or ROUNDS)) for a in used(ch)}
    preds["trees"] = apply_choice({a: final[a].predict(te_rows[FEATURES]) for a in final}, te_rows, ch)
    choices["trees"] = ch
    # deep (optional)
    if RUN_DEEP:
        net = deep_fit(fit_tr, va)
        ch = choose(deep_predict(net, va_rows), va_rows)
        net = deep_fit(tr, va)            # refit on every learning day; stopping still watches the last one
        preds["deep"] = apply_choice(deep_predict(net, te_rows), te_rows, ch)
        choices["deep"] = ch
    return te_rows, preds, choices, final


assert len(days) >= 3, "Need at least 3 days: two to learn and choose on, one to test."
rows_out, CHOICES = [], {}
for k in range(2, len(days)):
    tr_days, test_day = days[:k], days[k]
    te, preds, ch, last_trees = fit_all(tr_days, test_day)
    CHOICES[test_day] = ch
    for name, c in preds.items():
        e = c - te["y"].to_numpy()                            # predicted - actual; Passio's own: c = 0
        for (b, route), g in pd.DataFrame({"e": e, "lead": te["lead_s"].to_numpy(), "bin": te["bin"].astype(str).to_numpy(),
                                           "route": te["route_id"].astype(str).to_numpy()}).groupby(["bin", "route"]):
            rows_out.append({"test_day": test_day, "model": name, "bin": b, "route": route, **score(g["e"], g["lead"])})
    step(f"fold: learn {tr_days[0]} to {tr_days[-1]}, test {test_day}")
res = pd.DataFrame(rows_out)

## 9. Results, with the uncertainty that only more days can shrink
Per model and lead bin, pooled over the test days: band B (the target), the plain 10% rule, too late, the average
error, and a 95% range from resampling whole test days (the project's method). "vs Passio" is the band B
difference on the same rows; a real win needs that whole range above zero.

In [ ]:
def pooled(frame):
    s = frame[["n", "gross", "band_b", "pass_10", "late_1", "abs_err_sum"]].sum()
    ok = max(1, s["n"] - s["gross"])
    return {"n": int(s["n"]), "band_b": s["band_b"] / ok, "pass_10": s["pass_10"] / ok, "too_late": s["late_1"] / max(1, s["n"]),
            "mae_min": s["abs_err_sum"] / ok / 60, "gross": s["gross"] / max(1, s["n"])}


rng = np.random.default_rng(SEED)
test_days = sorted(res.test_day.unique())
# per (model, bin, day): [band_b, n - gross]; resample whole days 2,000 times for the band B difference vs Passio
day_sums = res.groupby(["model", "bin", "test_day"])[["band_b", "n", "gross"]].sum()
draws = rng.integers(0, len(test_days), size=(2000 if len(test_days) > 1 else 0, len(test_days)))


def day_arrays(model, b):
    t = day_sums.loc[(model, b)].reindex(test_days, fill_value=0)
    return t["band_b"].to_numpy(float), (t["n"] - t["gross"]).to_numpy(float)


table = []
for (model, b), g in res.groupby(["model", "bin"]):
    mb, mn = day_arrays(model, b)
    pb, pn = day_arrays("passio", b)
    sims = mb[draws].sum(1) / np.maximum(1, mn[draws].sum(1)) - pb[draws].sum(1) / np.maximum(1, pn[draws].sum(1))
    r = pooled(g)
    diff = r["band_b"] - pooled(res[(res.model == "passio") & (res.bin == b)])["band_b"]
    lo, hi = (np.percentile(sims, 2.5), np.percentile(sims, 97.5)) if len(sims) else (np.nan, np.nan)
    table.append({"model": model, "bin": b, **r, "vs_passio": diff, "ci_lo": lo, "ci_hi": hi})
tab = pd.DataFrame(table)
tab["bin"] = pd.Categorical(tab["bin"], BIN_NAMES, ordered=True)
order = [m for m in ["passio", "stop_shift", "trees", "deep"] if m in set(tab.model)]
tab["model"] = pd.Categorical(tab["model"], order, ordered=True)
tab = tab.sort_values(["bin", "model"])
show = tab.assign(**{c: (tab[c] * 100).round(1) for c in ["band_b", "pass_10", "too_late", "gross", "vs_passio", "ci_lo", "ci_hi"]},
                  mae_min=tab["mae_min"].round(2))
print(f"Test days: {', '.join(test_days)} (percent; vs_passio = band B difference, points)")
print(show[["bin", "model", "n", "band_b", "vs_passio", "ci_lo", "ci_hi", "pass_10", "too_late", "mae_min", "gross"]].to_string(index=False))
step("results")

## 10. Verdict against the ship gate
The project's gate for showing a new time to riders (GOAL-AND-METHOD 5.4): better than Passio in band B at
every lead from 2 to 30 minutes with the uncertainty range above zero, too late no more often than Passio, no
route more than 10% worse, measured on at least 10 test days. Until then a result is a first look.

In [ ]:
GATE_BINS = ["0-6", "6-10", "10-15", "15-20", "20-30"]
verdict = {}
for model in [m for m in order if m != "passio"]:
    t = tab[(tab.model == model) & tab["bin"].isin(GATE_BINS)].set_index("bin")
    p = tab[(tab.model == "passio") & tab["bin"].isin(GATE_BINS)].set_index("bin")
    wins = bool((t["ci_lo"] > 0).all())
    better = int((t["vs_passio"] > 0).sum())
    late_ok = bool((t["too_late"] <= p["too_late"] + 1e-9).all())
    rr = res[res.bin.isin(GATE_BINS)]
    worse = []
    for route, g in rr.groupby("route"):
        m, pp = pooled(g[g.model == model]), pooled(g[g.model == "passio"])
        if pp["n"] >= 50 and m["band_b"] < 0.9 * pp["band_b"]:
            worse.append(route)
    verdict[model] = {"bins_better": f"{better} of {len(GATE_BINS)}", "significant_every_bin": wins,
                      "too_late_not_worse": late_ok, "routes_more_than_10pct_worse": worse,
                      "test_days": len(test_days), "enough_days": len(test_days) >= 10}
    ok = wins and late_ok and not worse and len(test_days) >= 10
    print(f"{model:>10}: band B better in {better} of {len(GATE_BINS)} bins; significant everywhere: {wins}; "
          f"too late not worse: {late_ok}; routes > 10% worse: {worse or 'none'}; test days {len(test_days)} "
          f"-> {'PASSES the gate' if ok else 'not yet (first look)' if len(test_days) < 10 else 'fails the gate'}")
best = max([m for m in order if m != "passio"],
           key=lambda m: tab[(tab.model == m) & tab["bin"].isin(GATE_BINS)]["vs_passio"].mean())
print("Best on average over the 2 to 30 minute bins:", best)
step("verdict")

## 11. Save the results and a portable model
Writes the per-bin table (CSV), every number above (JSON), the trees of the last fold as LightGBM JSON, and the
per-stop shift as a plain lookup table. On the phone a model must run in plain JavaScript or Swift with no build
step: the lookup table can ship as is (a JSON file next to the schedule data), and a tree file can be walked by a
few lines of code. Nothing about the rider leaves the phone either way.

In [ ]:
show.to_csv(OUT / "results_by_bin.csv", index=False)
res.to_csv(OUT / "results_by_day_bin_route.csv", index=False)
meta = {"mode": MODE, "days": days, "test_days": test_days, "rows_learned_from": int(len(pool)),
        "features": FEATURES, "choices": CHOICES, "verdict": verdict, "best": best, "seconds_per_step": TIMES,
        "total_seconds": round(time.time() - T0, 1), "peak_memory_mb": round(PEAK[0]),
        "deep_model_run": bool(RUN_DEEP), "results": json.loads(show.to_json(orient="records"))}
(OUT / "metrics.json").write_text(json.dumps(meta, indent=1, default=str))
for a, m in last_trees.items():
    (OUT / f"trees_alpha_{a}.json").write_text(json.dumps(m.dump_model()))
lookup = shift_fit(pool, 0.2)[0].reset_index().rename(columns={"q": "shift_s"})
lookup.to_csv(OUT / "stop_shift_q20.csv", index=False)
print("Saved to", OUT, ":", ", ".join(sorted(p.name for p in OUT.iterdir())))
step("export")
print(f"Total {time.time() - T0:.0f} s, peak memory {PEAK[0]:,.0f} MB")

## Later, not now: a sequence model
With months of data, a model that reads the bus's last few stops in order (a small recurrent or transformer
network) could learn patterns a table cannot, such as a bus that has been slowing down for three stops. It is
not included because it needs far more days than exist, much more compute, and a way to run on the phone.
The trigger to try it: at least one full quarter of data, and the trees still clearly short of the goal at the
far leads.